# Phase 2 — Fixed-Weight Multi-Task Baseline (URI-CADS)

Shared encoder + kidney-segmentation head + global-pathology classification head, trained with a **simple fixed-weight loss sum** -- deliberately mirroring URI-CADS's own design choice (they state they used an unweighted sum "for simplicity").

**This is the baseline the optimizer (Phase 6) needs to beat.** Keep this notebook and its results untouched once trained -- it's your control condition.

## Setup: load cached zip + saved manifest (no re-parsing needed)

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import os

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/URI-CADS dataset'
DRIVE_ZIP_PATH = f'{DRIVE_PROJECT_ROOT}/uri_cads_archive.zip'
DRIVE_CHECKPOINTS_PATH = f'{DRIVE_PROJECT_ROOT}/checkpoints'
MANIFEST_SAVE_PATH = f'{DRIVE_PROJECT_ROOT}/manifest_with_splits.pkl'

LOCAL_ZIP_PATH = '/content/uri_cads_archive.zip'
LOCAL_DATA_DIR = '/content/uri_cads_data'
LOCAL_DATASET_DIR = os.path.join(LOCAL_DATA_DIR, 'bbdd_kidney')

os.makedirs(DRIVE_CHECKPOINTS_PATH, exist_ok=True)
print('Zip exists on Drive:', os.path.exists(DRIVE_ZIP_PATH))
print('Saved manifest exists:', os.path.exists(MANIFEST_SAVE_PATH))

Zip exists on Drive: True
Saved manifest exists: True


In [20]:
# Copy zip to local disk + unzip -- images/masks are still needed as actual files for training,
# even though we're skipping the label-parsing step by loading the saved manifest instead.
import shutil, zipfile, time

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

t0 = time.time()
shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
print(f'Copied zip in {time.time()-t0:.1f}s')

t0 = time.time()
with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zf:
    zf.extractall(LOCAL_DATA_DIR)
print(f'Unzipped in {time.time()-t0:.1f}s')
print('Contents:', os.listdir(LOCAL_DATASET_DIR))

Copied zip in 0.2s
Unzipped in 1.8s
Contents: ['labels', 'images', 'masks']


In [21]:
import pandas as pd

manifest = pd.read_pickle(MANIFEST_SAVE_PATH)

# image_path/mask_path were saved with LOCAL_DATASET_DIR baked in from the Phase 1 session --
# they should still resolve correctly here since we use the same local path convention.
print(manifest['split'].value_counts())
print(manifest.iloc[0][['image_path', 'mask_path', 'image_exists', 'mask_exists']])

split
train    1389
val       298
test      298
Name: count, dtype: int64
image_path      /content/uri_cads_data/bbdd_kidney/images/1.jpg
mask_path        /content/uri_cads_data/bbdd_kidney/masks/1.mat
image_exists                                               True
mask_exists                                                True
Name: 0, dtype: object


## Dataset class (same as Phase 1, duplicated here so this notebook is self-contained)

In [22]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import scipy.io

IMG_SIZE = (512, 512)


def load_kidney_mask(mat_path):
    data = scipy.io.loadmat(mat_path)
    mask = data['mask']
    return np.asarray(mask).astype(np.uint8)


def parse_local_lesions(extra_lines):
    lesions = []
    for line in extra_lines:
        if not line.startswith('Bounding box'):
            continue
        _, values = line.split(':', 1)
        nums = [int(v) for v in values.split()]
        if len(nums) != 5:
            continue
        x, y, w, h, pathology_id = nums
        lesions.append({'bbox': (x, y, x + w, y + h), 'pathology_id': pathology_id})
    return lesions


def bbox_to_coarse_mask(bbox, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)
    x_min, y_min, x_max, y_max = [int(round(v)) for v in bbox]
    mask[y_min:y_max, x_min:x_max] = 1
    return mask


class URICADSDataset(Dataset):
    def __init__(self, manifest_df, img_size=IMG_SIZE):
        self.manifest = manifest_df.reset_index(drop=True)
        self.img_size = img_size

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]
        target_h, target_w = self.img_size

        image = Image.open(row['image_path']).convert('RGB')
        orig_width, orig_height = image.size
        image = image.resize((target_w, target_h), Image.BILINEAR)
        image = np.array(image).astype(np.float32) / 255.0
        # normalize per your baseline convention: mean/std = 0.5 (not ImageNet stats)
        image = (image - 0.5) / 0.5

        kidney_mask = load_kidney_mask(row['mask_path'])
        kidney_mask = np.array(Image.fromarray(kidney_mask).resize((target_w, target_h), Image.NEAREST))

        global_diagnosis = int(row['global_diagnosis'])
        global_pathologies = torch.tensor(row['global_pathologies'], dtype=torch.float32)

        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float()
        kidney_mask_tensor = torch.from_numpy(kidney_mask).float()

        return {
            'image': image_tensor,
            'kidney_mask': kidney_mask_tensor,
            'global_diagnosis': torch.tensor(global_diagnosis, dtype=torch.float32),
            'global_pathologies': global_pathologies,
            'case_id': row['case_id'],
        }


train_ds = URICADSDataset(manifest[manifest['split'] == 'train'])
val_ds = URICADSDataset(manifest[manifest['split'] == 'val'])
test_ds = URICADSDataset(manifest[manifest['split'] == 'test'])

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

sample = train_ds[0]
print({k: (v.shape if hasattr(v, 'shape') else v) for k, v in sample.items()})

Train: 1389 | Val: 298 | Test: 298
{'image': torch.Size([3, 512, 512]), 'kidney_mask': torch.Size([512, 512]), 'global_diagnosis': torch.Size([]), 'global_pathologies': torch.Size([2]), 'case_id': '10'}


## Model: shared encoder + segmentation head + classification head

Deliberately lightweight (ResNet34 + simple upsampling decoder) rather than URI-CADS's heavy dual Mask-RCNN/Faster-RCNN setup -- this doubles as your first step toward the edge-deployability novelty angle, and trains far faster for iteration.

In [23]:
import torch.nn as nn
import torchvision

NUM_GLOBAL_PATHOLOGIES = len(manifest.iloc[0]['global_pathologies'])  # should be 2 (PCD, HC)
print('Num global pathology labels:', NUM_GLOBAL_PATHOLOGIES)


class MultiTaskNet(nn.Module):
    def __init__(self, num_global_pathologies=2, pretrained=True):
        super().__init__()
        backbone = torchvision.models.resnet34(weights='IMAGENET1K_V1' if pretrained else None)
        # keep everything up to the last conv block (drop avgpool + fc)
        self.encoder = nn.Sequential(*list(backbone.children())[:-2])  # -> B x 512 x H/32 x W/32

        # --- classification head: healthy/pathological + global pathologies ---
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.diagnosis_head = nn.Linear(512, 1)  # binary healthy vs pathological
        self.global_pathology_head = nn.Linear(512, num_global_pathologies)

        # --- segmentation head: simple upsampling decoder back to input resolution (x32 total) ---
        def up_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            )

        self.seg_decoder = nn.Sequential(
            up_block(512, 256),
            up_block(256, 128),
            up_block(128, 64),
            up_block(64, 32),
            up_block(32, 16),
            nn.Conv2d(16, 1, kernel_size=1),
        )

    def forward(self, x):
        feat = self.encoder(x)
        pooled = self.global_pool(feat).flatten(1)

        diagnosis_logits = self.diagnosis_head(pooled).squeeze(1)
        global_pathology_logits = self.global_pathology_head(pooled)
        seg_logits = self.seg_decoder(feat).squeeze(1)

        return diagnosis_logits, global_pathology_logits, seg_logits


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = MultiTaskNet(num_global_pathologies=NUM_GLOBAL_PATHOLOGIES).to(device)

Num global pathology labels: 2
Device: cuda


## Loss: fixed, equal weights (the baseline URI-CADS itself uses)

`L = L_diagnosis + L_global_pathology + L_seg` -- no tuning, no optimizer. This is intentional: it's the control condition Phase 6's optimized version has to outperform.

In [24]:
def dice_loss(pred_logits, target, eps=1e-6):
    pred = torch.sigmoid(pred_logits)
    pred_flat = pred.flatten(1)
    target_flat = target.flatten(1)
    intersection = (pred_flat * target_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()


bce_logits = nn.BCEWithLogitsLoss()


def compute_loss(diagnosis_logits, global_pathology_logits, seg_logits, batch):
    diagnosis_target = batch['global_diagnosis'].to(device)
    global_pathology_target = batch['global_pathologies'].to(device)
    seg_target = batch['kidney_mask'].to(device)

    loss_diagnosis = bce_logits(diagnosis_logits, diagnosis_target)
    loss_global = bce_logits(global_pathology_logits, global_pathology_target)
    loss_seg = dice_loss(seg_logits, seg_target)

    # FIXED, EQUAL weights -- matches URI-CADS's own stated design choice.
    total_loss = loss_diagnosis + loss_global + loss_seg
    return total_loss, {
        'diagnosis': loss_diagnosis.item(),
        'global_pathology': loss_global.item(),
        'seg': loss_seg.item(),
    }

## Training loop

In [25]:
BATCH_SIZE = 4       # adjust down if you hit CUDA OOM on the Colab GPU tier you have
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [26]:
import time
from tqdm.auto import tqdm


def dice_score(pred_logits, target, eps=1e-6):
    pred = (torch.sigmoid(pred_logits) > 0.5).float()
    pred_flat = pred.flatten(1)
    target_flat = target.flatten(1)
    intersection = (pred_flat * target_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    return ((2 * intersection + eps) / (union + eps)).mean().item()



def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    total_dice = 0.0
    n_batches = 0

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for batch in tqdm(loader, desc='train' if train else 'val'):
            images = batch['image'].to(device)
            diagnosis_logits, global_pathology_logits, seg_logits = model(images)
            loss, loss_breakdown = compute_loss(diagnosis_logits, global_pathology_logits, seg_logits, batch)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_dice += dice_score(seg_logits, batch['kidney_mask'].to(device))
            n_batches += 1

    return total_loss / n_batches, total_dice / n_batches

In [27]:
best_val_dice = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_dice = run_epoch(train_loader, train=True)
    val_loss, val_dice = run_epoch(val_loader, train=False)
    elapsed = time.time() - t0

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | '
          f'train_loss={train_loss:.4f} train_dice={train_dice:.4f} | '
          f'val_loss={val_loss:.4f} val_dice={val_dice:.4f} | {elapsed:.1f}s')

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        checkpoint_path = os.path.join(DRIVE_CHECKPOINTS_PATH, 'phase2_baseline_best.pt')
        torch.save(model.state_dict(), checkpoint_path)
        print(f'  -> new best val_dice={val_dice:.4f}, checkpoint saved to Drive')

train:   0%|          | 0/348 [00:00<?, ?it/s]

val:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch  1/15 | train_loss=1.8821 train_dice=0.2757 | val_loss=1.8362 val_dice=0.4663 | 73.4s
  -> new best val_dice=0.4663, checkpoint saved to Drive


train:   0%|          | 0/348 [00:00<?, ?it/s]

val:   0%|          | 0/75 [00:00<?, ?it/s]

...

val:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch 14/15 | train_loss=0.3450 train_dice=0.8823 | val_loss=1.4410 val_dice=0.8706 | 73.8s
  -> new best val_dice=0.8706, checkpoint saved to Drive


train:   0%|          | 0/348 [00:00<?, ?it/s]

val:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch 15/15 | train_loss=0.3312 train_dice=0.8898 | val_loss=1.8389 val_dice=0.8632 | 75.3s


## Test-set evaluation (Phase 2 baseline numbers -- your control condition)

In [28]:
from sklearn.metrics import roc_auc_score

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.load_state_dict(torch.load(os.path.join(DRIVE_CHECKPOINTS_PATH, 'phase2_baseline_best.pt')))
model.eval()

all_diag_probs, all_diag_targets = [], []
all_dice_scores = []

with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(device)
        diagnosis_logits, global_pathology_logits, seg_logits = model(images)

        all_diag_probs.extend(torch.sigmoid(diagnosis_logits).cpu().numpy())
        all_diag_targets.extend(batch['global_diagnosis'].numpy())
        all_dice_scores.append(dice_score(seg_logits, batch['kidney_mask'].to(device)))

test_auc = roc_auc_score(all_diag_targets, all_diag_probs)
test_dice = sum(all_dice_scores) / len(all_dice_scores)

print(f'PHASE 2 BASELINE RESULTS (fixed equal loss weights):')
print(f'  Healthy/Pathological AUC : {test_auc:.4f}   (URI-CADS reports 0.8741)')
print(f'  Kidney Segmentation Dice : {test_dice:.4f}')
print()
print('Save these numbers -- Phase 6 (optimized loss weights) needs to beat this AUC/Dice.')

PHASE 2 BASELINE RESULTS (fixed equal loss weights):
  Healthy/Pathological AUC : 0.9003   (URI-CADS reports 0.8741)
  Kidney Segmentation Dice : 0.8735

Save these numbers -- Phase 6 (optimized loss weights) needs to beat this AUC/Dice.
